<div style="text-align: center;">
  <span style="color: lightskyblue; font-weight: bold; font-size: 50px;">
    Test File for Depth Camera (D435i)
  </span>
</div>

<span style="color: magenta; font-weight: bold; font-size: 30px;">
Coordinate Information
</span>

<span style="color: white; font-weight: normal; font-size: 20px;">
pixel: (u,v), origin = top-left point, u: left-to-right, v: top-to-bottom
<br>
depth: z
<br>
3D position: (X,Y,Z), origin = top-left point, X: left-to-right, Y: top-to-bottom, Z=z(depth)
</span>

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import rby1_lib_ysw as ysw
import logging
import sys
import time
import pyrealsense2 as rs
import cv2

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Check recognition of D435i
</span>

In [2]:
ctx = rs.context()
devices = ctx.query_devices()

print("Number of RealSense devices:", len(devices))

for dev in devices:
    print("Name:", dev.get_info(rs.camera_info.name))
    print("Serial:", dev.get_info(rs.camera_info.serial_number))
    print("Firmware:", dev.get_info(rs.camera_info.firmware_version))

Number of RealSense devices: 1
Name: Intel RealSense D435I
Serial: 250122070658
Firmware: 5.15.1.55


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Robot
</span>

In [2]:
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot
robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Reset the Control Manager!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************





<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Power OFF
</span>

In [5]:
ysw.powerOFF(robot, robotConfig)

****************************** [powerOFF Starts] *******************************

[INFO] SUCCESS: Disable Grippers!
[INFO] SUCCESS: Disable the Control Manager!

****************************** [PowerOFF is Done] ******************************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Posture Change
</span>

In [3]:
Desired_Pos = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, Desired_Pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Vision
</span>

In [5]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640,480), fps=30)
d435i.initialize()

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Simple Test of D435i
</span>

In [3]:
# Simple test of RGB-D camera
# Click ESC: Terminate
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()
        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)
        
        if (d435i.rgb_on):
            d435i.imshow("color_image", color_image)
        if (d435i.depth_on):
            d435i.imshow("depth_image", depth_colorMap)
        
        key = d435i.waitKey(1)
        if key == 27:       # ESC key
            break
        
finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()
        

[INFO] D435i pipeline started


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Distance of center pixel
</span>

In [ ]:
# Get distance value of center pixel of image
# Click ESC: Terminate
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()
        
        if color_frame is None or depth_frame is None:
            continue
        
        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        h, w, _ = color_image.shape
        u = w // 2      # origin is left    
        v = h // 2      # origin is top

        distance = d435i.getDepth(depth_frame, (u, v))   # unit = [m]

        cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,                   
            f"Depth: {distance:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
3D position of center pixel based on Camera Reference Frame
</span>

In [4]:
# Show center 3D point position [x,y,z] in camera frame
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        h, w, _ = color_image.shape
        u = w // 2      # origin is left    
        v = h // 2      # origin is top 

        pos = d435i.get3DPos(depth_frame, (u,v))
        X, Y, Z = pos

        cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,
            f"X:{X:.3f}, Y:{Y:.3f}, Z:{Z:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
3D position of clicked point 
</span>

In [4]:
d435i.openWindowWithMouse("color_image")

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        if d435i.clicked_point is not None:
            u, v = d435i.clicked_point

            X, Y, Z = d435i.get3DPos(depth_frame, (u,v))

            cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
            cv2.putText(
                color_image,
                f"X:{X:.3f}, Y:{Y:.3f}, Z:{Z:.3f} m",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            print(f"Pixel=({u}, {v}), Camera 3D=({X:.3f}, {Y:.3f}, {Z:.3f}) m")

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

NameError: name 'd435i' is not defined

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Center 3D position of Colored Object
</span>

In [6]:
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = d435i.detectColorObjectContour(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,
            minimum_area=300                    # Can be changed
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"pixel={obj['pixel']}, "
                f"pos={np.round(obj['pos'], 3)}, "
                f"area={obj['area']:.1f}"
            )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)
        d435i.imshow("mask", mask)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started
[object 0] pixel=[624, 464], pos=[0.271 0.187 0.522], area=674.0
[object 1] pixel=[301, 314], pos=[-0.013  0.113  1.013], area=829.0
[object 3] pixel=[363, 91], pos=[ 0.184 -0.53   2.074], area=2244.0
[object 4] pixel=[561, 51], pos=[ 2.129 -1.65   5.133], area=6903.0


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

[object 0] pixel=[624, 465], pos=[0.269 0.187 0.519], area=651.5
[object 1] pixel=[300, 317], pos=[-0.016  0.122  1.053], area=871.5
[object 4] pixel=[363, 92], pos=[ 0.181 -0.518  2.04 ], area=2195.0
[object 5] pixel=[562, 51], pos=[ 2.12  -1.637  5.091], area=7333.0
[object 0] pixel=[624, 465], pos=[0.269 0.187 0.519], area=637.5
[object 1] pixel=[300, 317], pos=[-0.015  0.12   1.035], area=859.0
[object 4] pixel=[363, 92], pos=[ 0.18  -0.513  2.02 ], area=2195.5
[object 5] pixel=[562, 51], pos=[ 2.137 -1.65   5.133], area=7420.5
[object 0] pixel=[624, 465], pos=[0.269 0.187 0.519], area=635.0
[object 1] pixel=[300, 316], pos=[-0.015  0.119  1.035], area=863.5
[object 4] pixel=[364, 92], pos=[ 0.185 -0.518  2.04 ], area=2195.5
[object 5] pixel=[562, 51], pos=[ 2.12  -1.637  5.091], area=7454.5
[object 0] pixel=[624, 465], pos=[0.269 0.187 0.519], area=635.5
[object 1] pixel=[300, 317], pos=[-0.016  0.122  1.051], area=869.0
[object 4] pixel=[363, 92], pos=[ 0.178 -0.508  2.   ], area